
# Практикум по генераторам: “решай как на задании”

Этот ноутбук сделан так, чтобы ты **просто выполнял задания** (как на практике/доме) и по ходу **прочувствовал**, как работать с генераторами:
- где нужен `yield`;
- как тестировать себя `next()`, `list()`, `itertools.islice`;
- как строить ленивые пайплайны;
- как понять “почему не работает” по сообщениям тестов.

## Как пользоваться
1. Иди по порядку.
2. В ячейках с `# TODO` **пиши код**.
3. Запускай ячейку “Тесты” под заданием.
4. Если тесты падают — читай сообщение и проверяй маленькие примеры.

> Правило: **не делай `list()` от бесконечных генераторов.**



## 0) Вспомогательные штуки для проверки

Запусти следующую ячейку один раз.


In [1]:

import itertools
from typing import Callable

def check(name: str, fn: Callable[[], None]) -> None:
    """Запускает набор тестов и печатает результат."""
    try:
        fn()
    except AssertionError:
        print(f"❌ {name}: тест не пройден")
        raise
    except Exception as e:
        print(f"💥 {name}: ошибка ({type(e).__name__}: {e})")
        raise
    else:
        print(f"✅ {name}: всё ок")

def take_list(iterable, n: int):
    """Взять первые n элементов и вернуть списком (удобно для тестов)."""
    return list(itertools.islice(iterable, n))

print("Готово: инструменты проверки загружены.")


Готово: инструменты проверки загружены.



## 1) Задание: бесконечный генератор счётчика

Напиши генератор `count_from(start=0, step=1)`, который выдаёт:
- `start`, затем `start+step`, затем `start+2*step`, и так далее **без конца**.

Пример: `count_from(10, 3)` → `10, 13, 16, 19, ...`


In [4]:

def count_from(start: int = 0, step: int = 1):

    i = 0
    while True:
        yield start + step * i
        i += 1
    # TODO: реализуй генератор
    # Подсказка: while True: yield ...
    raise NotImplementedError


In [5]:

def _tests_count_from():
    g = count_from(10, 3)
    assert take_list(g, 5) == [10, 13, 16, 19, 22]

    g2 = count_from()
    assert take_list(g2, 4) == [0, 1, 2, 3]

    g3 = count_from(5, -2)
    assert take_list(g3, 4) == [5, 3, 1, -1]

check("Задание 1: count_from", _tests_count_from)


✅ Задание 1: count_from: всё ок



## 2) Задание: `take(iterable, n)` как генератор

Напиши генератор `take(iterable, n)`, который:
- выдаёт первые `n` элементов из `iterable`,
- затем останавливается.

`n` может быть 0.


In [24]:

def take(iterable, n: int):

    it = iter(iterable)
    for _ in range(n):
        yield next(it)
    # TODO: реализуй
    # Подсказка: it = iter(iterable); for _ in range(n): yield next(it)
    #raise NotImplementedError


In [25]:

def _tests_take():
    assert list(take(range(10), 3)) == [0, 1, 2]
    assert list(take([1,2,3], 0)) == []

    g = count_from(1, 1)
    assert list(take(g, 5)) == [1, 2, 3, 4, 5]

check("Задание 2: take", _tests_take)


✅ Задание 2: take: всё ок



## 3) Задание: `filter_map` — фильтр + преобразование (ленивый пайплайн)

Напиши генератор `filter_map(iterable, pred, func)`, который:
- берёт элементы из `iterable`,
- если `pred(x)` истинно — выдаёт `func(x)`,
- иначе пропускает.


In [39]:

def filter_map(iterable, pred, func):

    for x in iterable:
        if pred(x):
            yield func(x)

In [40]:

def _tests_filter_map():
    it = filter_map(range(10), lambda x: x % 2 == 0, lambda x: x*x)
    assert list(it) == [0, 4, 16, 36, 64]

    it2 = filter_map(count_from(1, 1), lambda x: x % 3 == 0, lambda x: x + 100)
    assert list(take(it2, 3)) == [103, 106, 109]

check("Задание 3: filter_map", _tests_filter_map)


✅ Задание 3: filter_map: всё ок



## 4) Задание: `chunked(iterable, size)` — порезать поток на чанки

Генератор должен выдавать списки-чанки длиной `size`.
Последний чанк может быть короче.

`size` должен быть > 0, иначе `ValueError`.


In [68]:

def chunked(iterable, size: int):

    if size <= 0:
        raise ValueError
        
    new_list = []
    i = 0
    for x in iterable:
        new_list.append(x)
        i += 1
        if i >= size:
            yield new_list 
            new_list = []
            i = 0
            
    if len(new_list) != 0:
        yield new_list

In [69]:

def _tests_chunked():
    assert list(chunked([1,2,3,4,5,6,7], 3)) == [[1,2,3], [4,5,6], [7]]
    assert list(chunked(range(6), 2)) == [[0,1], [2,3], [4,5]]

    try:
        list(chunked([1,2], 0))
        assert False, "Должен быть ValueError при size<=0"
    except ValueError:
        pass

check("Задание 4: chunked", _tests_chunked)


✅ Задание 4: chunked: всё ок



## 5) Задание: читать файл построчно и пропускать пустые строки

Напиши генератор `read_nonempty_lines(path)`, который:
- читает файл построчно,
- делает `strip()`,
- пропускает пустые строки,
- выдаёт очищенные строки.


In [70]:

from pathlib import Path

demo_path = Path("demo_input.txt")
demo_path.write_text("\n".join([
    "  alpha  ",
    "",
    "beta",
    "   ",
    "gamma   ",
    ""
]), encoding="utf-8")

print("Файл создан:", demo_path)
print("Содержимое:")
print(demo_path.read_text(encoding="utf-8"))


Файл создан: demo_input.txt
Содержимое:
  alpha  

beta
   
gamma   



In [73]:

def read_nonempty_lines(path):

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            
            if line:
                yield line
    

In [74]:

def _tests_read_nonempty_lines():
    assert list(read_nonempty_lines(demo_path)) == ["alpha", "beta", "gamma"]

check("Задание 5: read_nonempty_lines", _tests_read_nonempty_lines)


✅ Задание 5: read_nonempty_lines: всё ок



## 6) Задание: `moving_average()` через `send()` (корутина)

Сделай корутину `moving_average()`, которая:
- при каждом `.send(x)` принимает число `x`,
- возвращает текущее среднее.

Пример:
- `ma = moving_average(); next(ma)` (prime)
- `ma.send(10)` → `10.0`
- `ma.send(20)` → `15.0`
- `ma.send(30)` → `20.0`


In [85]:

def moving_average():

    values = []
    while True:
        med =sum(values) / len(values) if values else None
        value = yield  med
        values.append(value)
    # TODO: реализуй корутину
    # Подсказка:
    # values = []
    # while True:
    #     x = yield (sum(values)/len(values)) if values else None
    #     values.append(x)


In [86]:

def _tests_moving_average():
    ma = moving_average()
    _ = next(ma)  # prime

    r1 = ma.send(10)
    r2 = ma.send(20)
    r3 = ma.send(30)

    assert abs(r1 - 10.0) < 1e-9
    assert abs(r2 - 15.0) < 1e-9
    assert abs(r3 - 20.0) < 1e-9

check("Задание 6: moving_average(send)", _tests_moving_average)


✅ Задание 6: moving_average(send): всё ок



## 7) Финальный мини-челлендж: собрать пайплайн “из кубиков”

Используя **свои** функции выше, собери пайплайн:

1) бесконечный счётчик `count_from(1, 1)`  
2) оставить числа, которые делятся на 7  
3) заменить `x -> x*x`  
4) взять первые 10 результатов

Ожидается:
$$
7^2, 14^2, 21^2, \dots, 70^2
$$


In [93]:

# TODO: собери pipeline
# Подсказка: filter_map(count_from(...), pred, func) и потом take(..., 10)
pipeline = None

def pipeline():
    n = 10
    yield_counter = 0
    i = 0

    while i < n:
        i += 1
        yield (7*i)**2
        
pipeline = pipeline()



def _tests_pipeline():
    res = list(pipeline)
    expected = [(7*k)**2 for k in range(1, 11)]
    assert res == expected, (res, expected)

check("Задание 7: финальный пайплайн", _tests_pipeline)


✅ Задание 7: финальный пайплайн: всё ок



## 8) Что делать, если тесты падают

- Поставь маленький тест:
  - `g = твоя_функция(...)`
  - `print(next(g))`, `print(take_list(g, 5))`
- Помни: генераторы **одноразовые**
- На бесконечных потоках тестируй только первые элементы:
  - `take_list(...)` или `itertools.islice(...)`
